# Notebook 1 — Dataset Construction

Builds the instruction-tuning dataset from seed examples + GPT-4 synthetic generation.

**Input:** `data/raw/seed_examples.jsonl` (20 hand-crafted examples)

**Output:** `data/processed/train.jsonl` and `data/processed/test.jsonl`

In [ ]:
import sys, json
sys.path.append('../src')
from data_utils import load_jsonl, save_jsonl

seed = load_jsonl('../data/raw/seed_examples.jsonl')
print(f'{len(seed)} seed examples loaded')
print(json.dumps(seed[0], indent=2))

## Step 2 — Validate seed examples

In [ ]:
from data_utils import is_valid_policy

errors = []
for i, ex in enumerate(seed):
    if not all(k in ex for k in ['instruction', 'input', 'output']):
        errors.append(f'Example {i}: missing required keys')
    elif not is_valid_policy(ex['output'].get('policy', {})):
        errors.append(f'Example {i}: invalid policy structure')

if errors:
    for e in errors: print(f'ERROR: {e}')
else:
    print('All seed examples valid')

## Step 3 — Synthetic generation with Gemini 1.5 Flash

Uses seed examples as few-shot context. Review all outputs before saving.

Requires `GEMINI_API_KEY` in `.env` (project root).

In [ ]:
import os
import json
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv('../.env')
client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

GEMINI_MODEL = 'gemini-flash-lite-latest'  # alias — always resolves to latest lite flash

SYSTEM_INSTRUCTION = """You generate AWS IAM policy training examples.
Each example must be a valid JSON object with keys: instruction, input, output.
The output must contain: policy (valid AWS IAM JSON), nist_controls (list), risk_notes (string).
Follow the exact same structure as the examples provided."""

FEW_SHOT = json.dumps(seed[:3], indent=2)

def generate_synthetic_batch(n=10, service_hint=''):
    user_msg = f"""Here are example training records:
{FEW_SHOT}

Generate {n} NEW examples following the same structure.
Focus on: {service_hint or 'varied AWS services (S3, EC2, RDS, Lambda, IAM, KMS, SQS, SNS)'}.
Return a JSON array of {n} objects. No explanation, just the JSON array.
"""
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=user_msg,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            response_mime_type='application/json',
            temperature=0.8,
        )
    )
    raw = json.loads(response.text)
    if isinstance(raw, list):
        return raw
    return next(v for v in raw.values() if isinstance(v, list))

# Generate first batch — review before proceeding to bulk generation
synthetic = generate_synthetic_batch(n=10, service_hint='ECS, CloudWatch, Secrets Manager')
print(f'{len(synthetic)} synthetic examples generated')
print(json.dumps(synthetic[0], indent=2))

## Step 3b — Bulk generation (~500 examples)

Run this cell after confirming the single batch above looks good.  
Generates 10 themes × 5 batches × 10 examples = **500 synthetic examples**.  
Saves progress to `data/raw/synthetic_batch_01.jsonl` after each theme.

In [ ]:
# Load each source (edit files to remove bad examples before running this cell)
approved_synthetic = load_jsonl('../data/raw/synthetic_batch_01.jsonl')
approved_scraped   = load_jsonl('../data/raw/scraped_aws_docs.jsonl')

all_examples = seed + approved_synthetic + approved_scraped

print(f'Seeds:     {len(seed):4d}')
print(f'Synthetic: {len(approved_synthetic):4d}')
print(f'Scraped:   {len(approved_scraped):4d}')
print(f'Total:     {len(all_examples):4d}')

## Step 3c — Scrape real AWS documentation policies

Pulls actual IAM policy JSON from 12 AWS service documentation pages (S3, EC2, Lambda, RDS, KMS, …).  
Gemini then adds the plain-English `input` and NIST control mappings.  
These examples are grounded in AWS-official, production-tested policy patterns.

In [ ]:
from scrape_aws_docs import scrape_all

print('Scraping AWS documentation pages...')
raw_scraped = scrape_all()
print(f'\nTotal raw policy blocks found: {len(raw_scraped)}')

In [ ]:
def enrich_scraped_example(raw: dict) -> dict:
    """Ask Gemini to generate input text + NIST mappings for a real AWS policy."""
    prompt = f"""Convert this real AWS documentation IAM policy into a training record.

Service: {raw['service']}
Title: {raw['heading']}
Context: {raw['description']}
Policy:
{json.dumps(raw['policy'], indent=2)}

Return a JSON object with exactly these keys:
- "instruction": "Generate an AWS IAM policy for the following requirement"
- "input": a plain-English access control requirement (2-3 sentences) that would lead a practitioner to write exactly this policy
- "output": {{
    "policy": <the exact policy JSON above, unchanged>,
    "nist_controls": [relevant NIST SP 800-53 rev5 control IDs, e.g. "AC-3", "AC-6", "SC-28"],
    "risk_notes": "one sentence on the key security consideration for this policy"
  }}

Return only the JSON object, no explanation."""

    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type='application/json',
            temperature=0.2,
        )
    )
    return json.loads(response.text)


enriched_scraped = []
for i, raw in enumerate(raw_scraped):
    try:
        record = enrich_scraped_example(raw)
        enriched_scraped.append(record)
        print(f'[{i+1}/{len(raw_scraped)}] {raw["service"]:20s} {raw["heading"][:55]}')
        time.sleep(5)
    except Exception as e:
        print(f'[{i+1}/{len(raw_scraped)}] ERROR ({raw["service"]}): {e}')
        time.sleep(15)

save_jsonl(enriched_scraped, '../data/raw/scraped_aws_docs.jsonl')
print(f'\n{len(enriched_scraped)} enriched examples saved to data/raw/scraped_aws_docs.jsonl')

## Step 4 — Manual review

Scroll through synthetic examples. Remove any with hallucinated ARNs, wildcard abuse, or incorrect NIST mappings.

In [ ]:
# Save synthetic to raw for manual review before merging
save_jsonl(synthetic, '../data/raw/synthetic_batch_01.jsonl')

# After review: load approved synthetic + seeds
approved_synthetic = load_jsonl('../data/raw/synthetic_batch_01.jsonl')  # edit file to remove bad examples
all_examples = seed + approved_synthetic
print(f'Total after merge: {len(all_examples)}')

## Step 5 — Train/test split and save

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(all_examples, test_size=0.1, random_state=42)

save_jsonl(train, '../data/processed/train.jsonl')
save_jsonl(test, '../data/processed/test.jsonl')

print(f'Train: {len(train)} | Test: {len(test)}')
print('Dataset ready — proceed to notebook 02_finetune.ipynb')